In [ ]:
! pip install ultralytics

# Weighted Dataloader to fix class imbalance

In [ ]:
from ultralytics import YOLO
from ultralytics.data.dataset import YOLODataset
import ultralytics.data.build as build
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from collections import Counter

In [ ]:
class YOLOWeightedDataset(YOLODataset):
    def __init__(self, *args, mode="train", **kwargs):
        """
        Initialize the dataset with weights
        Args: class_weights (list or numpy array): A list or array of weights corresponding to each class.
        """
        # Inherit
        super().__init__(*args, **kwargs)
        self.train_mode = "train" in self.prefix

        # Count frequency of each class in the entire dataset 
        self.count_instances()
        # self.counts = [300, 200, 10]
        class_weights = np.sum(self.counts) / (self.counts + 1e-6)  # gives higher weight to weaker classes
        # Aggregate class weights with mean to combine weights for "balanced view" of objects
        self.agg_func = np.max
        # convert class_weights to numpy array for vectorized operations
        self.class_weights = np.array(class_weights)
        # self.weights = [sample_1 wt, sample_2 wt]
        # Assign weights to each sample
        self.weights = self.assign_weights()
        # Assign probabilities based on weights
        self.probabilities = self.calculate_probabilities()

    def count_instances(self):
        """
        Count class occurrences across the dataset.
        """
        counts = Counter()

        for label in self.labels:
            cls = label["cls"].reshape(-1).astype(int)
            for c in cls:
                counts[c] += 1

        # safe fallback for number of classes
        nc = getattr(self, "nc", None)
        if nc is None:
            nc = self.data["nc"]

        self.counts = np.array([counts.get(i, 0) for i in range(nc)])

    def assign_weights(self):
        """
       Calculate the aggregated weight for each label based on class weights.

       Returns:
           list: A list of aggregated weights corresponding to each label.
       """

        weights = []  # store final weight for each sample
        for label in self.labels:
            cls = label['cls'].reshape(-1).astype(int)

            # Give a default weight to the background class
            if cls.size == 0:
                weights.append(0.5)
                continue

            # weight = np.mean(self.class_weights[cls]) - balanced importance
            # weight = np.max(self.class_weights[cls]) - focus on rarest class
            weight = self.agg_func(self.class_weights[cls])
            weights.append(weight)  # combined image sample's weight

        weights = np.array(weights)
        weights = weights / weights.mean()
        return weights


    def calculate_probabilities(self):
        """
        Calculate and store the sampling probabilities based on the weights.
    
        Returns:
            list: A list of sampling probabilities corresponding to each label.
        """
        total_weight = self.weights.sum()
        probabilities = self.weights / total_weight
        return probabilities
    
    
    def __getitem__(self, index):
        """
        Return transformed label information based on the sampled index.
        """
        # Only for training
        if not self.train_mode:
            return self.transforms(self.get_image_and_label(index))
        else:
            index = np.random.choice(len(self.labels), p=self.probabilities)
            return self.transforms(self.get_image_and_label(index))

In [ ]:
# Import libraries
import os
import shutil
import yaml
import json

In [ ]:
!pip install pycocotools

In [ ]:
from pycocotools.coco import COCO

train_ann_path = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_train2017.json"
val_ann_path = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_val2017.json"
train_images_dir = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"
val_images_dir = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017"
output_labels_train_dir = "/kaggle/working/dataset/labels/train"
output_labels_val_dir = "/kaggle/working/dataset/labels/val"

# Create output directories
os.makedirs(output_labels_train_dir, exist_ok=True)
os.makedirs(output_labels_val_dir, exist_ok=True)

nc = 5

In [ ]:
# Convert bbox coordinates to yolo convention
def convert_bbox_to_yolo(img_w, img_h, bbox):
    x, y, w, h = bbox
    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h
    w /= img_w
    h /= img_h
    return x_center, y_center, w, h

In [ ]:
def process_coco_annotations(annotation_path, output_labels_dir, images_dir):
    # Initialize COCO API
    coco = COCO(annotation_path)
    
    # Define our target classes
    target_classes = ['person', 'bicycle', 'car', 'bus', 'truck']
    cat_ids = coco.getCatIds(catNms=target_classes)
    
    # Create a mapping from COCO category_id to YOLO class index (0-4)
    # This ensures 'person' is 0, 'bicycle' is 1, etc., regardless of COCO's internal IDs
    coco_id_to_yolo_idx = {coco_id: i for i, coco_id in enumerate(cat_ids)}
    
    # Get all image IDs that contain any of our target classes
    img_ids = set()
    for cat_id in cat_ids:
        class_ids = coco.getImgIds(catIds=[cat_id])
        img_ids.update(class_ids)
    img_ids = list(img_ids)
    
    processed_filenames = []

    print(f"Processing {len(img_ids)} images containing target classes")

    for img_id in img_ids:
        img_info = coco.loadImgs([img_id])[0]
        img_filename = img_info['file_name']
        img_w, img_h = img_info['width'], img_info['height']
        
        # Check if the image file actually exists before creating a label for it
        if not os.path.exists(os.path.join(images_dir, img_filename)):
            continue

        # Load annotations for this image, filtered by our categories
        ann_ids = coco.getAnnIds(imgIds=[img_id], catIds=cat_ids)
        anns = coco.loadAnns(ann_ids)
        
        if not anns:
            continue

        # Prepare label file path (e.g., 000000000139.txt)
        label_file_name = os.path.splitext(img_filename)[0] + ".txt"
        label_path = os.path.join(output_labels_dir, label_file_name)
        
        valid_labels_found = False
        with open(label_path, 'w') as f:
            for ann in anns:
                if 'bbox' in ann:
                    # COCO bbox is [x_min, y_min, width, height]
                    # Convert to YOLO [x_center, y_center, width, height] normalized
                    yolo_bbox = convert_bbox_to_yolo(img_w, img_h, ann['bbox'])
                    
                    # Map the original COCO category_id to our 0-4 range
                    yolo_class_idx = coco_id_to_yolo_idx[ann['category_id']]
                    
                    line = f"{yolo_class_idx} {' '.join(f'{x:.6f}' for x in yolo_bbox)}\n"
                    f.write(line)
                    valid_labels_found = True
        
        if valid_labels_found:
            processed_filenames.append(img_filename)

    return processed_filenames

In [ ]:
# Process both training and validation annotations
train_set = process_coco_annotations(train_ann_path, output_labels_train_dir, train_images_dir)
val_set = process_coco_annotations(val_ann_path, output_labels_val_dir, val_images_dir)

print(f"COCO Train Set: {len(train_set)} images")
print(f"COCO Val Set: {len(val_set)} images")

In [ ]:
# Organize dataset in structure suitable for yolov8

train_images_dir = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017"
val_images_dir = "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017"
output_train_images_dir = "/kaggle/working/dataset/images/train"
output_val_images_dir = "/kaggle/working/dataset/images/val"

os.makedirs(output_train_images_dir, exist_ok=True)
os.makedirs(output_val_images_dir, exist_ok=True)

print("Organize training images and respective labels")
for img_filename in train_set:
    img_path = os.path.join(train_images_dir, img_filename)
    symlink_path = os.path.join(output_train_images_dir, img_filename)
    if not os.path.exists(symlink_path):
        os.symlink(img_path, symlink_path)

print("Organize validation images and respective labels")
for img_filename in val_set:
    img_path = os.path.join(val_images_dir, img_filename)
    symlink_path = os.path.join(output_val_images_dir, img_filename)
    if not os.path.exists(symlink_path):
        os.symlink(img_path, symlink_path)

for folder in [output_labels_train_dir, output_labels_val_dir]:
    img_folder = folder.replace("labels", "images")
    valid_images = {os.path.splitext(f)[0] for f in os.listdir(img_folder)}
    
    for label_file in os.listdir(folder):
        label_name = os.path.splitext(label_file)[0]
        if label_name not in valid_images:
            os.remove(os.path.join(folder, label_file))

In [ ]:
# Create data.yaml
data_yaml = """
train: /kaggle/working/dataset/images/train
val: /kaggle/working/dataset/images/val
nc: 5
names:
  0: 'person'
  1: 'bicycle'
  2: 'car'
  3: 'bus'
  4: 'truck'
"""
with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

# Model Training

In [ ]:
build.YOLODataset = YOLOWeightedDataset

# Load pretrained model
model = YOLO("yolov8m.pt")

# Train the model
results = model.train(data='/kaggle/working/dataset/data.yaml', epochs=20, imgsz=512, batch=16, cache=True, copy_paste=0.1, mosaic=0.7, mixup=0.05)

In [ ]:
# Verify that the model used Weighted Dataset for training
model.trainer.train_loader.dataset

In [ ]:
img = cv2.imread("runs/detect/train/train_batch0.jpg")[...,::-1]
plt.imshow(img)

# Check class balance

In [ ]:
def verify_class_balance(dataset, num_samples=1000):
    """
    Verifies whether the __getitem__ method in the YOLOWeightedDataset class returns a balanced class output.

    Args:
        dataset: An instance of YOLOWeightedDataset.
        num_samples: Number of samples to draw from the dataset.

    Returns:
        class_counts: A dictionary containing the class counts.
    """
    all_labels = []
    num_samples = min(len(dataset.labels), num_samples)

    if dataset.train_mode:
        choices = np.random.choice(len(dataset.labels), size=num_samples, p=dataset.probabilities)
    else:
        choices = np.random.choice(len(dataset.labels), size=num_samples, replace=False)

    for _ in range(num_samples):
        sample = dataset[0]  # triggers weighted sampling internally
        cls = sample['cls']
        all_labels.extend(cls.reshape(-1).astype(int))

    class_counts = Counter(all_labels)
    return class_counts

def plot_class_balance(weighted_cnts, unweighted_cnts, class_names):
    """
    Plots the comparison of class distribution between training and validation modes.

    Args:
        weighted_cnts: A dictionary containing the class counts in weighted mode.
        unweighted_cnts: A dictionary containing the class counts in unweighted mode.
        class_names: A list of class names.
    """
    classes = range(len(class_names))
    weighted_values = [weighted_cnts.get(c, 0) for c in classes]
    unweighted_values = [unweighted_cnts.get(c, 0) for c in classes]

    width = 0.35  # Bar width

    fig, ax = plt.subplots()
    ax.bar(classes, unweighted_values, width, label='Normal mode')
    ax.bar([c + width for c in classes], weighted_values, width, label='Weighted Mode')

    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.set_title('Class Distribution in Normal vs Weighted Modes')
    ax.set_xticks([c + width / 2 for c in classes])
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.legend()

    plt.show()

In [ ]:
last_model = "runs/detect/train/weights/last.pt"

In [ ]:
build.YOLODataset = YOLODataset

metrics = model.val(data='/kaggle/working/dataset/data.yaml')
print("Validation Results:")

In [ ]:
best_model = "runs/detect/train/weights/best.pt"

In [ ]:
test_images_path = '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/test2017'
test_images = os.listdir(test_images_path)[:100]
test_images_full_paths = [os.path.join(test_images_path, img) for img in test_images]

test_results = model.predict(source=test_images_full_paths, save=True, save_txt=False)

from pathlib import Path

output_dir = Path("/kaggle/working/detect")  
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Results saved in: {output_dir}")

# Visualize Training Results

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

train_dir = "/kaggle/working/runs/detect/train" 

train_images = [
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "confusion_matrix.png",
    "labels.jpg",
    "results.png",
]

num_images = len(train_images)
cols = 2 
rows = (num_images + cols - 1) // cols  

fig, axes = plt.subplots(rows, cols, figsize=(15, 15))
axes = axes.flatten()  

for i, img_name in enumerate(train_images):
    img_path = os.path.join(train_dir, img_name)
    if os.path.exists(img_path):
        print(f"Displaying: {img_name}")
        img = plt.imread(img_path)
        axes[i].imshow(img)
        axes[i].axis('off')  # Hide the axes
        axes[i].set_title(img_name)
    else:
        print(f"File not found: {img_name}")
        axes[i].axis('off')  
for j in range(i + 1, rows * cols):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

Visualize Validation Results

In [ ]:
val_dir = "/kaggle/working/runs/detect/val"  

val_images = [
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "confusion_matrix.png",
    "val_batch0_pred.jpg",
    "val_batch1_pred.jpg",
]

for img_name in val_images:
    img_path = os.path.join(val_dir, img_name)
    if os.path.exists(img_path):
        print(f"Displaying: {img_name}")
        display(Image(filename=img_path))
    else:
        print(f"File not found: {img_name}")

# Visualizing Predicted Images

In [ ]:
predict_dir = "/kaggle/working/runs/detect/val2"
predicted_images = [
    img for img in os.listdir(predict_dir) if img.endswith(('.png', '.jpg', '.jpeg'))
]
for img_name in predicted_images[:30]: 
    img_path = os.path.join(predict_dir, img_name)
    if os.path.exists(img_path):
        print(f"Displaying: {img_name}")
        display(Image(filename=img_path))
    else:
        print(f"File not found: {img_name}")